# Advanced Problems with Solutions: Reference Counting

This notebook develops a deeper understanding of **reference counting** in Python.

## Learning goals
- Understand what a reference count represents.
- Compare `sys.getrefcount(...)` with lower-level inspection using `ctypes`.
- Predict how assignment, function calls, containers, and scope affect reference counts.
- Recognize why reference-count experiments can be fragile and implementation-dependent.

## Important notes
- The techniques shown here are mainly for **learning and debugging**.
- In CPython, object lifetime is heavily influenced by reference counting, but Python programmers usually do not manage memory manually.
- Some behaviors shown here are **CPython-specific** and should not be treated as portable Python guarantees.
- Small integer caching, string interning, notebook state, and temporary references can all affect the exact numbers you observe.

## Refresher Setup

In [1]:
import ctypes
import sys

def ref_count(address):
    return ctypes.c_long.from_address(address).value

## Refresher Example

We start with one list and inspect its reference count in two ways.

In [2]:
my_var = [1, 2, 3, 4]

print('id(my_var) =', id(my_var))
print('ref_count(id(my_var)) =', ref_count(id(my_var)))
print('sys.getrefcount(my_var) =', sys.getrefcount(my_var))

id(my_var) = 3099335939520
ref_count(id(my_var)) = 1
sys.getrefcount(my_var) = 2


You will usually see that `sys.getrefcount(my_var)` is **one higher** than `ref_count(id(my_var))`.

Why? Because `sys.getrefcount(my_var)` must receive `my_var` as an argument, which temporarily creates one more reference during the call.

---

# Problems

## Problem 1 — Why Are the Two Counts Different?

Run the code and explain why the two counts differ.

1. Which function tends to report the "actual current" count more directly?
2. Why is the built-in helper often off by 1?
3. Should you rely on exact counts in every environment?

In [3]:
sample = ['x', 'y', 'z']

print('ref_count(id(sample)) =', ref_count(id(sample)))
print('sys.getrefcount(sample) =', sys.getrefcount(sample))

ref_count(id(sample)) = 1
sys.getrefcount(sample) = 2


### Solution 1

`ref_count(id(sample))` inspects the count directly from the object's address.

`sys.getrefcount(sample)` usually returns a value that is **one larger**, because the function call itself temporarily holds a reference to `sample` while evaluating the argument.

So the difference is not a contradiction. It reflects the measurement method.

**Best practice:** use `sys.getrefcount(...)` only as a learning/debugging tool, and remember its result is typically inflated by 1.

**Caution:** exact counts may vary because notebook machinery, temporary expressions, or implementation details can introduce extra references.

## Problem 2 — Assignment Increases the Reference Count

Predict what happens after creating an alias.

1. Do `a` and `b` point to the same object?
2. How should the reference count change?
3. Why does assignment not copy the list?

In [4]:
a = [10, 20, 30]
before = ref_count(id(a))

b = a
after = ref_count(id(a))

print('id(a) =', id(a))
print('id(b) =', id(b))
print('a is b ->', a is b)
print('count before =', before)
print('count after  =', after)

id(a) = 3099335854784
id(b) = 3099335854784
a is b -> True
count before = 1
count after  = 2


### Solution 2

`b = a` does **not** copy the list. It creates a second reference to the same list object.

Therefore:
- `id(a)` and `id(b)` are the same.
- `a is b` is `True`.
- The reference count for that list should increase by about 1.

**Core idea:** assignment binds a name to an existing object unless a new object is explicitly created.

## Problem 3 — Removing a Reference

What happens when one alias is rebound to `None`?

Predict the reference count before and after rebinding.

In [5]:
values = [1, 2, 3]
other = values

count_before = ref_count(id(values))
other = None
count_after = ref_count(id(values))

print('count before rebinding other =', count_before)
print('count after rebinding other  =', count_after)

count before rebinding other = 2
count after rebinding other  = 1


### Solution 3

Originally, both `values` and `other` refer to the same list object.

When we execute:

```python
other = None
```

`other` stops referring to that list, so the list loses one reference.

As a result, the reference count should decrease by about 1.

**Core idea:** rebinding a variable can reduce the old object's reference count if that variable was one of the references to it.

## Problem 4 — Passing an Object to a Function

A function parameter is also a reference.

Predict what happens to the count inside the function.

In [6]:
def show_count(obj):
    print('Inside function:')
    print('id(obj) =', id(obj))
    print('ref_count(id(obj)) =', ref_count(id(obj)))
    print('sys.getrefcount(obj) =', sys.getrefcount(obj))

items = ['a', 'b']
print('Outside before call:', ref_count(id(items)))
show_count(items)
print('Outside after call :', ref_count(id(items)))

Outside before call: 1
Inside function:
id(obj) = 3099335947584
ref_count(id(obj)) = 2
sys.getrefcount(obj) = 3
Outside after call : 1


### Solution 4

When `items` is passed into `show_count`, the parameter `obj` becomes another reference to the same object.

So inside the function, the reference count is usually higher than outside the function.

Also, calling `sys.getrefcount(obj)` adds yet another temporary reference during the call, so that reported number is typically one higher than the lower-level count.

After the function returns, the local parameter `obj` disappears, so the count outside usually drops back.

**Core idea:** function arguments temporarily increase the number of references to an object.

## Problem 5 — Container References Count Too

A list can hold a reference to another object. Predict how that affects reference counting.

In [7]:
target = {'course': 'python'}
before = ref_count(id(target))

holder = [target]
after = ref_count(id(target))

print('before =', before)
print('after  =', after)
print('holder[0] is target ->', holder[0] is target)

before = 1
after  = 2
holder[0] is target -> True


### Solution 5

Yes, container elements count as references too.

When `holder = [target]` is created, the new list stores a reference to the dictionary object referenced by `target`.

So the dictionary's reference count should increase by about 1.

`holder[0] is target` is `True` because both names lead to the same dictionary object.

**Core idea:** references are not only created by variable names; container slots also hold references.

## Problem 6 — Mutation Does Not Necessarily Change the Reference Count

Does changing the contents of a list change how many references point to that list?

In [8]:
numbers = [1, 2, 3]
alias = numbers

before = ref_count(id(numbers))
numbers.append(4)
after = ref_count(id(numbers))

print('numbers =', numbers)
print('alias   =', alias)
print('before count =', before)
print('after count  =', after)
print('numbers is alias ->', numbers is alias)

numbers = [1, 2, 3, 4]
alias   = [1, 2, 3, 4]
before count = 2
after count  = 2
numbers is alias -> True


### Solution 6

Appending to the list mutates the existing object in place.

That changes the list's contents, but it does **not** normally change how many references point to that list.

So the count usually stays the same before and after `numbers.append(4)`.

**Core idea:** mutation changes an object's state, not necessarily the number of names or containers pointing to it.

## Problem 7 — Rebinding vs. Mutation

Compare these two operations carefully.

Which one changes the reference count of the original object?

In [9]:
data = [1, 2]
alias = data

count_before = ref_count(id(data))
data = data + [3]
count_after_rebind = ref_count(id(alias))

print('After rebinding data:')
print('data  =', data,  'id(data) =', id(data))
print('alias =', alias, 'id(alias) =', id(alias))
print('count of original object now =', count_after_rebind)

alias.append(99)
print('\nAfter alias.append(99):')
print('data  =', data)
print('alias =', alias)
print('count of alias object =', ref_count(id(alias)))

After rebinding data:
data  = [1, 2, 3] id(data) = 3099335942656
alias = [1, 2] id(alias) = 3099335942400
count of original object now = 1

After alias.append(99):
data  = [1, 2, 3]
alias = [1, 2, 99]
count of alias object = 1


### Solution 7

Initially, `data` and `alias` both reference the same list.

But this line:

```python
data = data + [3]
```

creates a **new list** and rebinds `data` to it. The original list remains referenced by `alias`.

So the original list loses one reference when `data` is rebound.

Later, `alias.append(99)` mutates that original list in place. That mutation changes contents, not identity.

**Core idea:** rebinding can change reference counts; mutation usually does not.

## Problem 8 — Reference Counts and Nested Structures

Suppose an inner object is shared by multiple outer containers. Predict what happens.

In [10]:
inner = ['shared']
count_1 = ref_count(id(inner))

outer1 = [inner]
count_2 = ref_count(id(inner))

outer2 = {'key': inner}
count_3 = ref_count(id(inner))

print('initial count        =', count_1)
print('after list container  =', count_2)
print('after dict container  =', count_3)
print('outer1[0] is inner ->', outer1[0] is inner)
print("outer2['key'] is inner ->", outer2['key'] is inner)

initial count        = 1
after list container  = 2
after dict container  = 3
outer1[0] is inner -> True
outer2['key'] is inner -> True


### Solution 8

Both `outer1` and `outer2` store references to the same inner list.

So each additional container slot that points to `inner` should increase its reference count.

This is why nested and shared data structures can keep objects alive longer than expected.

**Core idea:** any live object that stores a reference helps keep the referenced object alive.

## Problem 9 — Temporary References in Expressions

The exact count may be surprisingly hard to predict during expression evaluation.

Observe the output and explain why counts taken mid-expression may not be stable.

In [11]:
obj = [1, 2, 3]

print('baseline:', ref_count(id(obj)))
print('during getrefcount call:', sys.getrefcount(obj))
print('after call:', ref_count(id(obj)))

baseline: 1
during getrefcount call: 2
after call: 1


### Solution 9

The count seen during an expression may include temporary references introduced by:
- function arguments
- local variables
- the interactive environment
- debugging or notebook machinery

That is why reference-count experiments are useful for intuition, but exact numbers should be interpreted carefully.

**Best practice:** focus on how counts change qualitatively rather than depending on exact values in interactive environments.

## Problem 10 — What Keeps an Object Alive?

Will this object still exist after the function returns?

In [12]:
def make_and_return():
    local_obj = ['alive']
    print('Inside function count =', ref_count(id(local_obj)))
    return local_obj

result = make_and_return()
print('Outside function count =', ref_count(id(result)))

Inside function count = 1
Outside function count = 1


### Solution 10

Inside the function, `local_obj` has at least one reference from the local variable.

When the function returns `local_obj`, the caller receives a reference to the same object and stores it in `result`.

So the object remains alive after the function ends because `result` still refers to it.

If nothing referred to that object after the function returned, its reference count would drop, and in CPython it would typically become eligible for immediate cleanup.

**Core idea:** objects stay alive as long as something still references them.

## Problem 11 — A Common Misconception About `del`

Does `del` delete the object itself, or does it delete a reference?

Predict what remains after `del name1`.

In [13]:
name1 = [7, 8, 9]
name2 = name1

print('count before del =', ref_count(id(name1)))
del name1

print('name2 =', name2)
print('count after del  =', ref_count(id(name2)))

count before del = 2
name2 = [7, 8, 9]
count after del  = 1


### Solution 11

`del name1` removes the binding of the name `name1`; it does not necessarily destroy the object immediately.

Because `name2` still refers to the list, the object remains alive.

So after `del name1`:
- the object still exists
- `name2` still works
- the reference count should decrease by about 1

**Core idea:** `del` removes a reference, not automatically the underlying object.

## Problem 12 — Putting It All Together

For each numbered line, explain how the reference count of `shared` should change.

In [14]:
shared = ['core']                        # 1
a = shared                              # 2
b = [shared, shared]                    # 3
c = {'x': shared}                       # 4
a = None                                # 5
b = None                                # 6
c = None                                # 7

print('shared =', shared)
print('final count =', ref_count(id(shared)))

shared = ['core']
final count = 1


### Solution 12

Step by step:

1. `shared = ['core']`
   - A new list is created and bound to `shared`.

2. `a = shared`
   - One more reference is added.

3. `b = [shared, shared]`
   - The list `b` stores **two references** to the same object, so the count increases by about 2.

4. `c = {'x': shared}`
   - The dictionary stores one more reference.

5. `a = None`
   - The old reference from `a` is removed.

6. `b = None`
   - The container `b` is no longer referenced, so its two references to `shared` are removed as well.

7. `c = None`
   - The dictionary reference is removed.

At the end, only the name `shared` should still be referencing the object, so the final count should be back near its starting level.

**Core idea:** reference counts depend on all live references, including repeated references from the same container.

# Summary of Best Practices

1. Treat reference counting as a mental model, not something you usually manipulate directly.
2. Remember that `sys.getrefcount(x)` is typically one higher because of the function argument itself.
3. Assignment creates references; it does not copy objects unless a copy is explicitly made.
4. Containers store references too.
5. Mutation usually changes object contents, not reference counts.
6. Rebinding and `del` can reduce reference counts by removing bindings.
7. Exact counts in notebooks or interactive sessions may include temporary extra references.
8. Use these tools for understanding CPython behavior, not for normal application logic.

# Optional Challenge Problems

Try these without looking up the answer first:

1. Why can the same object appear multiple times in one container and increase the count multiple times?
2. How would a shallow copy of a container affect the reference counts of inner objects?
3. Why might exact reference counts differ between a script and a Jupyter notebook?
4. Why is reference counting not the full story for memory management in Python?